# Brain Tumor Classification - Model Training

## Dual-Path Neural Network with Cross-Modal Gating

```
Path A (Report):
  BioClinicalBERT (fine-tune top 2 layers) → CLS token → 288d

Path B (Tabular):
  Clinical (3d) + SI one-hot (24d) + Location embedding (10d)
  + Hierarchical location (30d: region/hemisphere/lobe)
  + Cross-modal ratios (25d) + Missing flags (4d) → MLP → 272d

CrossModalGate: Bidirectional gated interaction between paths
  - Single shared gate: g = sigmoid(r_key * t_key)
  - report_out = report + g * tabular_signal
  - tab_out = tabular + g * report_signal

Classifier: 1120d → 256d → 128d → 5
  (288 + 272 + 288 + 272 = 1120d)
```

**Architecture features:**
- BioClinicalBERT for clinical text understanding
- Hierarchical location encoding (robust to novel locations in test)
- Cross-modal ratios from radiomics (T1c/T1, T2F/T2, etc.)
- Gated cross-modal interaction (adaptive weighting per sample)
- SWA + cosine annealing + warmup for robust training

**Training configuration:**
- 5-fold stratified cross-validation (random_state=42)
- CrossEntropy loss with label smoothing (0.05) and sqrt class weights (alpha=0.5)
- BERT LR: 1.5e-6 (MAIN_LR=3e-5, BERT_LR_SCALE=0.05)
- Early stopping (patience=25)
- Stochastic Weight Averaging (SWA) from epoch 70%  
  


*AI-assisted tools were used for code debugging and figure visualization. All experimental design, model architecture decisions, and analysis were conducted by the authors.*


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Brain Tumor Classify"

In [ ]:
import sys
import os
sys.path.insert(0, os.getcwd())

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertModel
from data_loader import get_all_data, parse_location_hierarchy

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## 1. Load Data


In [ ]:
data = get_all_data('kaggle-dataset', pca_dim=128, test_data_dir='new_test')

# Label encoder & class info
train_label = data['train']['label']
val_label = data['val']['label']
label_encoder = data['label_encoder']
NUM_CLASSES = len(label_encoder.classes_)
print(f"Classes: {list(label_encoder.classes_)}")
print(f"Num classes: {NUM_CLASSES}")

# Class distribution analysis (for reference only, not for weighting)
print(f"\nClass distribution in training set:")
for i, cls in enumerate(label_encoder.classes_):
    n = (np.array(train_label) == i).sum()
    pct = 100 * n / len(train_label)
    print(f"  {cls:45s} n={n:4d}  ({pct:.1f}%)")

# Note: We do NOT use class weights because:
# 1. Kaggle uses Micro-F1 which is dominated by majority classes
# 2. Weighting hurts performance on majority classes
# 3. This reduces overall Micro-F1 score
class_weights = None
print(f"\nUsing UNWEIGHTED loss (optimizing for Micro-F1)")

---
## 2. Constants

In [ ]:
# Shared constants
MODALITIES = ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']
BATCH_SIZE = 32

EMBED_DIM = 256
ATTN_LAYERS = 3
ATTN_HEADS = 4

# SI column name mapping
IMG_DIM = 128  # PCA-reduced image dimension (128 or 256, was 2048)

SI_COL_MAPPING = {
    'ax_t1': 'Signal Intensity (T1)',
    'ax_t1c': 'Signal Intensity (T1c)',
    'ax_t2': 'Signal Intensity (T2)',
    'ax_t2f': 'Signal Intensity (T2-FLAIR)',
}

print(f"Modalities: {MODALITIES}")

Modalities: ['ax_t1', 'ax_t1c', 'ax_t2', 'ax_t2f']


---
## 3. BERT Report Encoding (Pre-compute)

In [ ]:
# Build case_id -> report text mapping
def build_report_map(data_dict):
    reports_df = data_dict['report']
    return dict(zip(reports_df['case_id'].astype(int), reports_df['report'].fillna('')))

train_reports = build_report_map(data['train'])
val_reports = build_report_map(data['val'])
test_reports = build_report_map(data['test'])

# Tokenize
MAX_LEN = 128
print("Loading ClinicalBERT tokenizer...")
tokenizer = BertTokenizer.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')

def tokenize_reports(case_ids, report_map):
    texts = [report_map.get(cid, '') for cid in case_ids]
    encoded = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
    return encoded['input_ids'], encoded['attention_mask']

# Get case_ids from merged dataframes (in order)
train_case_ids = data['train']['merged']['case_id'].tolist()
val_case_ids = data['val']['merged']['case_id'].tolist()
test_case_ids = data['test']['merged']['case_id'].tolist()

train_input_ids, train_attention_mask = tokenize_reports(train_case_ids, train_reports)
val_input_ids, val_attention_mask = tokenize_reports(val_case_ids, val_reports)
test_input_ids, test_attention_mask = tokenize_reports(test_case_ids, test_reports)

print(f"Train reports tokenized: {train_input_ids.shape}")
print(f"Val reports tokenized: {val_input_ids.shape}")
print(f"Test reports tokenized: {test_input_ids.shape}")

Loading ClinicalBERT tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

Train reports tokenized: torch.Size([1983, 128])
Val reports tokenized: torch.Size([283, 128])
Test reports tokenized: torch.Size([378, 128])


In [ ]:
# ============================================================
# BERT Embeddings: Pre-compute (OPTIONAL - only if NOT fine-tuning BERT)
# ============================================================
# If fine-tuning BERT (cell-14 uses ReportEncoder with live BERT),
# skip this cell. Pre-computed embeddings are NOT needed.
#
# If you want to use frozen BERT (faster, less GPU memory),
# uncomment the code below and run it.

print("Skipping BERT pre-computation.")
print("BERT will be fine-tuned live during training (see ReportEncoder in cell-14).")
print("If you want frozen BERT, uncomment the code below.")

# --- Uncomment below for frozen BERT ---
# def compute_bert_embeddings(input_ids, attention_mask, device, batch_size=32):
#     bert_model = BertModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')
#     bert_model.eval()
#     bert_model.to(device)
#     all_embeddings = []
#     with torch.no_grad():
#         for i in range(0, len(input_ids), batch_size):
#             ids_batch = input_ids[i:i+batch_size].to(device)
#             mask_batch = attention_mask[i:i+batch_size].to(device)
#             outputs = bert_model(input_ids=ids_batch, attention_mask=mask_batch)
#             hidden = outputs.last_hidden_state
#             mask_exp = mask_batch.unsqueeze(-1).float()
#             pooled = (hidden * mask_exp).sum(dim=1) / mask_exp.sum(dim=1).clamp(min=1)
#             all_embeddings.append(pooled.cpu())
#     embeddings = torch.cat(all_embeddings, dim=0)
#     del bert_model
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#     return embeddings
#
# train_bert_emb = compute_bert_embeddings(train_input_ids, train_attention_mask, device)
# val_bert_emb = compute_bert_embeddings(val_input_ids, val_attention_mask, device)
# test_bert_emb = compute_bert_embeddings(test_input_ids, test_attention_mask, device)
# print(f"Train BERT embeddings: {train_bert_emb.shape}")
# print(f"Val BERT embeddings: {val_bert_emb.shape}")
# print(f"Test BERT embeddings: {test_bert_emb.shape}")



---
## 4. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader, Subset, ConcatDataset


class BrainTumorDataset(Dataset):
    """
    Dual-path architecture (NO Image):
      Path A (Report):
        report_input_ids:       np.array(128,) — BERT token IDs
        report_attention_mask:  np.array(128,) — BERT attention mask
      Path B (Tabular):
        clinical:       np.array(3,) — Sex, Age, Age_missing
        location:       int — label-encoded tumor location
        location_hierarchy: np.array(3,) — region, hemisphere, lobe
        si_per_mod:     dict[mod -> np.array(si_dim)]  — one-hot per modality
        si_unknown_mask: dict[mod -> np.array(1,)] — 1=known, 0=unknown
        cross_mod_ratios: np.array(25,) — T1c/T1, T2F/T2, T1/T2, deltas
        rad_missing:    np.array(4,) — per-modality missing flags
    """

    def __init__(self, data_dict, input_ids, attention_mask, case_ids, has_label=True):
        self.merged = data_dict['merged']
        self.has_label = has_label
        self.input_ids = input_ids        # (N, 128)
        self.attention_mask = attention_mask  # (N, 128)
        self.case_ids = case_ids
        self.case_to_idx = {cid: i for i, cid in enumerate(self.case_ids)}

        # SI columns per modality
        self.si_cols = {}
        for mod in MODALITIES:
            si_name = SI_COL_MAPPING[mod]
            self.si_cols[mod] = [c for c in self.merged.columns if c.startswith(si_name + '_')]

        # Radiomics per modality (for cross-modal ratios)
        self.rad_cols = {}
        for mod in MODALITIES:
            self.rad_cols[mod] = [c for c in self.merged.columns if c.startswith(f'{mod}_rad_')]

        self.clin_cols = ['Sex_enc', 'Age_clean', 'Age_missing']
        self.si_dim = len(self.si_cols[MODALITIES[0]])

    def __len__(self):
        return len(self.case_ids)

    def __getitem__(self, idx):
        case_id = self.case_ids[idx]
        row = self.merged[self.merged['case_id'] == case_id].iloc[0]

        # Radiomics per modality (raw, for cross-modal ratio computation)
        rad_raw = {}
        for mod in MODALITIES:
            cols = self.rad_cols[mod]
            rad_raw[mod] = (
                row[cols].values.astype(np.float32) if cols
                else np.zeros(5, dtype=np.float32)
            )

        # Cross-modal ratios: 5 features × 5 ratio types = 25 features
        eps = 1e-8
        ratios = []
        for i in range(5):
            ratios.append(np.log(np.clip((rad_raw['ax_t1c'][i] + eps) / (rad_raw['ax_t1'][i] + eps), eps, 100.0)))
            ratios.append(np.log(np.clip((rad_raw['ax_t2f'][i] + eps) / (rad_raw['ax_t2'][i] + eps), eps, 100.0)))
            ratios.append(np.log(np.clip((rad_raw['ax_t1'][i] + eps) / (rad_raw['ax_t2'][i] + eps), eps, 100.0)))
            ratios.append(np.clip(rad_raw['ax_t1c'][i] - rad_raw['ax_t1'][i], -100, 100))
            ratios.append(np.clip(rad_raw['ax_t2f'][i] - rad_raw['ax_t2'][i], -100, 100))
        cross_mod_ratios = np.array(ratios, dtype=np.float32)

        # Per-modality missing flags
        rad_missing = np.array([
            1.0 if row.get(f'ax_t1_missing', 0) else 0.0,
            1.0 if row.get(f'ax_t1c_missing', 0) else 0.0,
            1.0 if row.get(f'ax_t2_missing', 0) else 0.0,
            1.0 if row.get(f'ax_t2f_missing', 0) else 0.0,
        ], dtype=np.float32)

        # SI per modality + unknown mask
        si_per_mod = {}
        si_unknown_mask = {}
        for mod in MODALITIES:
            cols = self.si_cols[mod]
            si_vec = (
                row[cols].values.astype(np.float32) if cols
                else np.zeros(self.si_dim, dtype=np.float32)
            )
            si_per_mod[mod] = si_vec
            unknown_col = [c for c in cols if c.endswith('_unknown')]
            is_unknown = 1.0 if (unknown_col and row[unknown_col[0]] == 1) else 0.0
            si_unknown_mask[mod] = np.array([1.0 - is_unknown], dtype=np.float32)  # 1=known, 0=unknown

        # Clinical
        clin_vals = row[self.clin_cols].values.astype(np.float32)
        location_val = int(row.get('Location_enc', 0))

        # Hierarchical location
        loc_text = str(row.get('Tumor_Location_raw', ''))
        region_enc, hemisphere_enc, lobe_enc = parse_location_hierarchy(loc_text)
        location_hierarchy = np.array([region_enc, hemisphere_enc, lobe_enc], dtype=np.int64)

        # BERT tokenized input
        bert_idx = self.case_to_idx[case_id]
        report_input_ids = self.input_ids[bert_idx].numpy().astype(np.int64)
        report_attention_mask = self.attention_mask[bert_idx].numpy().astype(np.int64)

        base = {
            'report_input_ids': report_input_ids,
            'report_attention_mask': report_attention_mask,
            'clinical': clin_vals,
            'location': location_val,
            'location_hierarchy': location_hierarchy,
            'si_per_mod': si_per_mod,
            'si_unknown_mask': si_unknown_mask,
            'cross_mod_ratios': cross_mod_ratios,
            'rad_missing': rad_missing,
        }
        if self.has_label:
            base['label'] = int(row['label'])
        return base


# Instantiate datasets
train_ds = BrainTumorDataset(data['train'], train_input_ids, train_attention_mask, train_case_ids, has_label=True)
val_ds = BrainTumorDataset(data['val'], val_input_ids, val_attention_mask, val_case_ids, has_label=True)
test_ds = BrainTumorDataset(data['test'], test_input_ids, test_attention_mask, test_case_ids, has_label=False)

# Verify feature dimensions
sample = train_ds[0]
print("Dual-path feature dims (first sample):")
print(f"  Path A (Report): IDs={sample['report_input_ids'].shape}, Mask={sample['report_attention_mask'].shape}")
print(f"  Path B (Tabular):")
print(f"    Clinical: {sample['clinical'].shape}")
print(f"    Location: scalar + hierarchy={sample['location_hierarchy'].shape}")
print(f"    SI per modality: {train_ds.si_dim}d each x {len(MODALITIES)}")
print(f"    Cross-modal ratios: {sample['cross_mod_ratios'].shape}")
print(f"    Radiomics missing flags: {sample['rad_missing'].shape}")
print(f"  Total tabular input: ~64d")



In [ ]:
# Collate function & DataLoaders
def collate_fn(batch):
    si_per_mod = {mod: [] for mod in MODALITIES}
    si_unknown_mask = {mod: [] for mod in MODALITIES}
    for b in batch:
        for mod in MODALITIES:
            si_per_mod[mod].append(b['si_per_mod'][mod])
            si_unknown_mask[mod].append(b['si_unknown_mask'][mod])

    result = {
        'report_input_ids': np.stack([b['report_input_ids'] for b in batch]),
        'report_attention_mask': np.stack([b['report_attention_mask'] for b in batch]),
        'si_per_mod': {mod: np.stack(si_per_mod[mod]) for mod in MODALITIES},
        'si_unknown_mask': {mod: np.stack(si_unknown_mask[mod]) for mod in MODALITIES},
        'clinical': np.stack([b['clinical'] for b in batch]),
        'location': np.array([b['location'] for b in batch]),
        'location_hierarchy': np.stack([b['location_hierarchy'] for b in batch]),
        'cross_mod_ratios': np.stack([b['cross_mod_ratios'] for b in batch]),
        'rad_missing': np.stack([b['rad_missing'] for b in batch]),
    }
    if 'label' in batch[0]:
        result['label'] = np.array([b['label'] for b in batch])
    return result


def to_device(batch, device):
    out = {
        'report_input_ids': torch.from_numpy(batch['report_input_ids']).to(device).long(),
        'report_attention_mask': torch.from_numpy(batch['report_attention_mask']).to(device).long(),
        'si_per_mod': {mod: torch.from_numpy(v).to(device) for mod, v in batch['si_per_mod'].items()},
        'si_unknown_mask': {mod: torch.from_numpy(v).to(device) for mod, v in batch['si_unknown_mask'].items()},
        'clinical': torch.from_numpy(batch['clinical']).to(device),
        'location': torch.from_numpy(batch['location']).to(device).long(),
        'location_hierarchy': torch.from_numpy(batch['location_hierarchy']).to(device).long(),
        'cross_mod_ratios': torch.from_numpy(batch['cross_mod_ratios']).to(device),
        'rad_missing': torch.from_numpy(batch['rad_missing']).to(device),
    }
    if 'label' in batch:
        out['label'] = torch.from_numpy(batch['label']).to(device)
    return out


# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)

print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")



Train: 62 batches | Val: 9 | Test: 12


## 5. Model Architecture

```
Dual-Path Brain Tumor Classifier:

Path A: ReportEncoder
  BioClinicalBERT (layers 10-11 fine-tuned, 9 frozen)
    → CLS token → 768d → Linear → 288d
    + Residual connection

Path B: TabularEncoder
  Clinical (3d) → 32d
  SI one-hot (24d) → 80d  
  Location (10d) + Hierarchy (30d)
  Radiomics ratios (25d) + Missing flags (4d)
    → 206d → MLP → 272d

CrossModalGate:
  r_key = Linear(report, 64)
  t_key = Linear(tabular, 64)
  g = sigmoid(r_key * t_key)  # Single shared gate
  report_out = norm(report + g * to_report(tabular))
  tab_out = norm(tabular + g * to_report(report))

Classifier:
  [report, tabular, report_att, tab_att] = 1120d
    → 256d → 128d → 5
```

**Key dimensions:**
- Report embedding: 288d
- Tabular embedding: 272d
- Cross-attended: 288d + 272d
- Classifier input: 1120d


In [ ]:
# ============================================================
# Dual-Path Brain Tumor Classifier
#
# Architecture:
#   Path A: BioClinicalBERT → 288d (fine-tune layers 10-11)
#   Path B: Clinical + SI + Location + Hierarchy + Ratios → 272d
#   CrossModalGate: Single shared gate for bidirectional interaction
#   Classifier: 1120d → 256d → 128d → 5
#
# Total trainable params: ~112M (BERT) + ~1M model = ~113M
# ============================================================

FREEZE_BERT_LAYERS = 10       # Fine-tune top 2 layers (9 frozen, fine-tune 10-11)
REPORT_DROPOUT = 0.15        # Light -- report is the strongest signal
TABULAR_DROPOUT = 0.30      # Moderate -- tabular features
REPORT_DIM = 288             # Slightly larger
TABULAR_DIM = 272            # Slightly larger


class ReportEncoder(nn.Module):
    def __init__(self, output_dim=288, freeze_bert_layers=10, dropout=0.20):
        super().__init__()
        self.bert = BertModel.from_pretrained('emilyalsentzer/Bio_ClinicalBERT')

        for param in self.bert.embeddings.parameters():
            param.requires_grad = False
        for param in self.bert.pooler.parameters():
            param.requires_grad = False

        for i, layer in enumerate(self.bert.encoder.layer):
            if i < freeze_bert_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        # CLS token + projection (simple, works well for short reports)
        self.proj = nn.Sequential(
            nn.Linear(768, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.residual = nn.Linear(768, output_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        return self.proj(cls_output) + self.residual(cls_output)


class TabularEncoder(nn.Module):
    def __init__(self, num_locations=636, location_emb_dim=10, si_dim=6,
                 num_modalities=4, output_dim=272,
                 num_regions=4, num_hemispheres=5, num_lobes=10,
                 dropout=0.25):
        super().__init__()

        self.location_emb = nn.Embedding(num_locations, location_emb_dim)
        self.region_emb = nn.Embedding(num_regions, 10)
        self.hemisphere_emb = nn.Embedding(num_hemispheres, 10)
        self.lobe_emb = nn.Embedding(num_lobes, 10)

        si_total_dim = si_dim * num_modalities
        self.si_proj = nn.Sequential(
            nn.Linear(si_total_dim, 80),
            nn.LayerNorm(80),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        rad_input_dim = 25 + 4
        self.rad_proj = nn.Sequential(
            nn.Linear(rad_input_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.clin_proj = nn.Sequential(
            nn.Linear(3, 32),
            nn.LayerNorm(32),
            nn.GELU(),
        )

        combined_dim = 32 + 80 + 64 + location_emb_dim + 30
        self.tabular_mlp = nn.Sequential(
            nn.Linear(combined_dim, 320),
            nn.LayerNorm(320),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(320, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, clinical, location, location_hierarchy, si_per_mod, si_unknown_mask, cross_mod_ratios, rad_missing):
        loc_emb = self.location_emb(location)

        region_emb = self.region_emb(location_hierarchy[:, 0])
        hemisphere_emb = self.hemisphere_emb(location_hierarchy[:, 1])
        lobe_emb = self.lobe_emb(location_hierarchy[:, 2])
        loc_hierarchy_emb = torch.cat([region_emb, hemisphere_emb, lobe_emb], dim=-1)

        si_list = [si_per_mod[mod] for mod in MODALITIES]
        si_flat = torch.cat(si_list, dim=-1)
        si_emb = self.si_proj(si_flat)

        unknown_masks = [si_unknown_mask[mod] for mod in MODALITIES]
        unknown_mask = torch.cat(unknown_masks, dim=-1)
        si_unknown = (unknown_mask.mean(dim=-1, keepdim=True) < 0.5).float()
        si_emb = si_emb * (1 - si_unknown * 0.5)

        rad_input = torch.cat([cross_mod_ratios, rad_missing], dim=-1)
        rad_emb = self.rad_proj(rad_input)
        clin_emb = self.clin_proj(clinical)

        combined = torch.cat([clin_emb, si_emb, rad_emb, loc_emb, loc_hierarchy_emb], dim=-1)
        return self.tabular_mlp(combined)


class CrossModalGate(nn.Module):
    """Lightweight gating between Report and Tabular embeddings (~25K params)."""
    def __init__(self, report_dim=288, tabular_dim=272, dropout=0.2):
        super().__init__()

        # Project to common dim for interaction
        common_dim = 64
        self.report_key = nn.Linear(report_dim, common_dim)
        self.tabular_key = nn.Linear(tabular_dim, common_dim)

        # Gate: how much to blend from the other modality
        self.gate = nn.Sequential(
            nn.Linear(common_dim, 1),
            nn.Sigmoid(),
        )

        # Cross-projection (report → tabular space, tabular → report space)
        self.report_to_tab = nn.Linear(report_dim, tabular_dim)
        self.tab_to_report = nn.Linear(tabular_dim, report_dim)

        self.norm_r = nn.LayerNorm(report_dim)
        self.norm_t = nn.LayerNorm(tabular_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, report_emb, tab_emb):
        # Compute interaction
        r_key = self.report_key(report_emb)   # (B, 64)
        t_key = self.tabular_key(tab_emb)     # (B, 64)

        # Gate based on similarity
        g = self.gate(r_key * t_key)  # (B, 1) -- how much cross-modal info to use

        # Cross-modal signal
        r2t = self.dropout(self.report_to_tab(report_emb))   # (B, tab_dim)
        t2r = self.dropout(self.tab_to_report(tab_emb))      # (B, rep_dim)

        # Gated blend
        report_out = self.norm_r(report_emb + g * t2r)
        tab_out = self.norm_t(tab_emb + g * r2t)

        return report_out, tab_out


class DualPathBrainTumorClassifier(nn.Module):
    def __init__(self, num_classes=5, si_dim=6, num_locations=636,
                 report_dim=288, tabular_dim=272,
                 num_regions=4, num_hemispheres=5, num_lobes=10,
                 freeze_bert_layers=10, report_dropout=0.15, tabular_dropout=0.30,
                 use_cross_attn=True):
        super().__init__()
        self.use_cross_attn = use_cross_attn

        self.report_encoder = ReportEncoder(
            output_dim=report_dim,
            freeze_bert_layers=freeze_bert_layers,
            dropout=report_dropout,
        )
        self.tabular_encoder = TabularEncoder(
            num_locations=num_locations,
            location_emb_dim=10,
            si_dim=si_dim,
            num_modalities=len(MODALITIES),
            output_dim=tabular_dim,
            num_regions=num_regions,
            num_hemispheres=num_hemispheres,
            num_lobes=num_lobes,
            dropout=tabular_dropout,
        )

        if use_cross_attn:
            self.cross_attn = CrossModalGate(
                report_dim=report_dim,
                tabular_dim=tabular_dim,
                dropout=0.2,
            )

        fused_dim = report_dim + tabular_dim
        if use_cross_attn:
            fused_dim += report_dim + tabular_dim

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_classes),
        )

    def forward(self, report_input_ids, report_attention_mask,
                clinical, location, location_hierarchy, si_per_mod, si_unknown_mask,
                cross_mod_ratios, rad_missing):
        report_emb = self.report_encoder(report_input_ids, report_attention_mask)
        tabular_emb = self.tabular_encoder(
            clinical, location, location_hierarchy, si_per_mod, si_unknown_mask,
            cross_mod_ratios, rad_missing
        )

        if self.use_cross_attn:
            report_att, tab_att = self.cross_attn(report_emb, tabular_emb)
            fused = torch.cat([report_emb, tabular_emb, report_att, tab_att], dim=-1)
        else:
            fused = torch.cat([report_emb, tabular_emb], dim=-1)

        logits = self.classifier(fused)
        return logits


NUM_LOCATIONS = len(data['loc_encoder'].classes_)
NUM_REGIONS = 4
NUM_HEMISPHERES = 5
NUM_LOBES = 10
si_dim = sample['si_per_mod'][MODALITIES[0]].shape[0]
print(f"SI dim: {si_dim}, Num locations: {NUM_LOCATIONS}")
print(f"Report dim: {REPORT_DIM}, Tabular dim: {TABULAR_DIM}")
print(f"BERT freeze layers: {FREEZE_BERT_LAYERS} (fine-tuning top {12-FREEZE_BERT_LAYERS} layers)")
print(f"Report dropout: {REPORT_DROPOUT}, Tabular dropout: {TABULAR_DROPOUT}")
print(f"Hierarchy: regions={NUM_REGIONS}, hemispheres={NUM_HEMISPHERES}, lobes={NUM_LOBES}")

model = DualPathBrainTumorClassifier(
    num_classes=NUM_CLASSES,
    si_dim=si_dim,
    num_locations=NUM_LOCATIONS,
    report_dim=REPORT_DIM,
    tabular_dim=TABULAR_DIM,
    num_regions=NUM_REGIONS,
    num_hemispheres=NUM_HEMISPHERES,
    num_lobes=NUM_LOBES,
    freeze_bert_layers=FREEZE_BERT_LAYERS,
    report_dropout=REPORT_DROPOUT,
    tabular_dropout=TABULAR_DROPOUT,
)
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: Total={total_params:,} | Trainable={trainable:,}")

print("\nPer-module params:")
for name, module in model.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {n:,}")
model = model.to(device)

---
## 6. Training Loop


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, fbeta_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from torch.optim.swa_utils import AveragedModel, SWALR
from tqdm.auto import tqdm
import warnings
import copy
import math
warnings.filterwarnings('ignore', message='.*torch.no_grad.*')

# ============================================================
# Improved Training Utilities (Unlimited Compute Edition)
#
# Key improvements:
#   1. COSINE ANNEALING scheduler instead of ReduceLROnPlateau
#   2. LESS AGGRESSIVE augmentation (noise 0.05, mask 0.1)
#   3. SQRT WEIGHTS (alpha=0.5) for gentle class balance
#   4. SWA from 70%
#   5. Better warmup schedule
# ============================================================
BERT_LR_SCALE = 0.05        # Higher BERT LR (was 0.05)
MAIN_LR = 3e-5            # Higher LR (was 2e-5)
NN_WEIGHT_ALPHA = 0.5     # 0.5*sqrt(balanced) -- gentle minority boost
LABEL_SMOOTHING = 0.05    # Less smoothing (was 0.08)
N_SPLITS = 5             # 5-fold for 2K dataset
SWA_START_FRAC = 0.70     # Start SWA earlier (was 0.75)
SWA_LR = 1e-6
TAB_NOISE_STD = 0.10      # Less noise (was 0.15)
MIXUP_ALPHA = 0.1         # Less mixup (was 0.2)
REPORT_MASK_PROB = 0.12   # Less masking (was 0.15)
WARMUP_EPOCHS = 3         # Gradual warmup


def f1_beta_micro(y_true, y_pred, beta=1.0):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='micro', beta=beta, zero_division=0
    )
    return f1


def evaluate_all_betas(y_true, y_pred, model_name="Model"):
    betas = [0.5, 1.0, 2.0]

    print(f"\n{model_name} - Multi-Metric Evaluation:")
    print(f"  {'Metric':>20s} | {'Score':>8s}")
    print(f"  {'-'*20}-|-{'-'*8}")

    micro_f1 = f1_beta_micro(y_true, y_pred, beta=1.0)
    print(f"  {'Micro-F1 (=Acc)':>20s} | {micro_f1:>8.4f}")

    print(f"\n  Macro-F1 (equal class weight) with beta sensitivity:")
    print(f"  {'Beta':>6s} | {'Macro-F1':>8s} | {'Description':20s}")
    print(f"  {'-'*6}-|-{'-'*8}-|-{'-'*20}")

    macro_results = {}
    for beta in betas:
        f1 = fbeta_score(y_true, y_pred, beta=beta, average='macro', zero_division=0)
        macro_results[beta] = f1
        desc = {0.5: "Precision weighted", 1.0: "Balanced F1", 2.0: "Recall weighted"}.get(beta, "")
        print(f"  {beta:>6.1f} | {f1:>8.4f} | {desc:20s}")

    per_class_acc = {}
    for i, cls in enumerate(label_encoder.classes_):
        mask = (y_true == i)
        if mask.sum() > 0:
            acc = (y_pred[mask] == i).mean()
            per_class_acc[cls] = acc

    print(f"\n  Per-class Accuracy:")
    for cls, acc in per_class_acc.items():
        print(f"    {cls}: {acc:.4f}")

    return {'micro': micro_f1, 'macro': macro_results}, per_class_acc


def compute_soft_weights(train_labels, alpha=0.5):
    """Compute sqrt class weights for soft balancing."""
    classes = np.unique(train_labels)
    cw_balanced = compute_class_weight('balanced', classes=classes, y=train_labels)
    cw_sqrt = np.sqrt(cw_balanced)
    uniform = np.ones_like(cw_sqrt)
    cw_final = (1 - alpha) * uniform + alpha * cw_sqrt
    cw_final = cw_final / cw_final.mean()

    print(f"\nClass weights (alpha={alpha}):")
    for i, cls in enumerate(label_encoder.classes_):
        n = (np.array(train_labels) == i).sum()
        print(f"  {cls:45s} n={n:4d}  weight={cw_final[i]:.3f}")

    return torch.tensor(cw_final, dtype=torch.float32).to(device)


def train_epoch(model, loader, optimizer, criterion, device, epoch, augment=True, scheduler=None):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch} Train")
    for batch in pbar:
        batch = to_device(batch, device)
        labels = batch['label']

        if augment:
            # Light augmentation
            aug_mask = batch['report_attention_mask'].clone()
            mask_prob = torch.rand_like(aug_mask.float()) < REPORT_MASK_PROB
            aug_mask = aug_mask * (~mask_prob | (batch['report_attention_mask'] == 0)).long()

            aug_ratios = batch['cross_mod_ratios'].clone()
            if TAB_NOISE_STD > 0:
                noise = torch.randn_like(aug_ratios) * TAB_NOISE_STD
                aug_ratios = aug_ratios + noise

            aug_clinical = batch['clinical'].clone()
            if TAB_NOISE_STD > 0:
                clin_noise = torch.randn_like(aug_clinical) * (TAB_NOISE_STD * 0.5)
                aug_clinical = aug_clinical + clin_noise
        else:
            aug_mask = batch['report_attention_mask']
            aug_ratios = batch['cross_mod_ratios']
            aug_clinical = batch['clinical']

        optimizer.zero_grad()

        logits = model(
            report_input_ids=batch['report_input_ids'],
            report_attention_mask=aug_mask,
            clinical=aug_clinical,
            location=batch['location'],
            location_hierarchy=batch['location_hierarchy'],
            si_per_mod=batch['si_per_mod'],
            si_unknown_mask=batch['si_unknown_mask'],
            cross_mod_ratios=aug_ratios,
            rad_missing=batch['rad_missing'],
        )

        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    micro_f1 = f1_beta_micro(all_labels, all_preds, beta=1.0)
    return total_loss / len(loader), micro_f1


def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            batch = to_device(batch, device)
            logits = model(
                report_input_ids=batch['report_input_ids'],
                report_attention_mask=batch['report_attention_mask'],
                clinical=batch['clinical'],
                location=batch['location'],
                location_hierarchy=batch['location_hierarchy'],
                si_per_mod=batch['si_per_mod'],
                si_unknown_mask=batch['si_unknown_mask'],
                cross_mod_ratios=batch['cross_mod_ratios'],
                rad_missing=batch['rad_missing'],
            )
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            if 'label' in batch:
                all_labels.extend(batch['label'].cpu().numpy())

    if all_labels:
        metrics, per_class_acc = evaluate_all_betas(
            np.array(all_labels), np.array(all_preds), "Neural Network"
        )
        return metrics, per_class_acc, all_preds, all_labels
    return None, None, all_preds, None


def average_checkpoints(checkpoint_list):
    avg_state = copy.deepcopy(checkpoint_list[0])
    for key in avg_state:
        for ckpt in checkpoint_list[1:]:
            avg_state[key] += ckpt[key]
        avg_state[key] = avg_state[key] / len(checkpoint_list)
    return avg_state


def train_model(model, train_loader, val_loader, epochs=300, lr=MAIN_LR, patience=30,
                weight_alpha=NN_WEIGHT_ALPHA, label_smoothing=LABEL_SMOOTHING,
                train_labels_for_weights=None):
    bert_params = []
    other_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'report_encoder.bert' in name:
            bert_params.append(param)
        else:
            other_params.append(param)

    optimizer = torch.optim.AdamW([
        {'params': bert_params, 'lr': lr * BERT_LR_SCALE, 'weight_decay': 0.01},
        {'params': other_params, 'lr': lr, 'weight_decay': 0.02},
    ])

    # Cosine annealing with warmup
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * WARMUP_EPOCHS

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # Loss with sqrt weights
    if weight_alpha > 0:
        labels_for_weights = train_labels_for_weights if train_labels_for_weights is not None else train_label
        weights = compute_soft_weights(labels_for_weights, alpha=weight_alpha)
        criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)
        print(f"  Using SQRT WEIGHTED loss (alpha={weight_alpha}, label_smoothing={label_smoothing})")
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        print(f"  Using UNWEIGHTED loss (label_smoothing={label_smoothing})")

    print(f"  Main LR={lr:.2e}, BERT LR={lr * BERT_LR_SCALE:.2e}")
    print(f"  Cosine annealing scheduler with {WARMUP_EPOCHS} warmup epochs")
    print(f"  Augmentation: report_mask={REPORT_MASK_PROB}, tab_noise={TAB_NOISE_STD}")

    best_val_f1 = 0
    best_state = None
    top_checkpoints = []
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device, epoch,
            augment=True, scheduler=scheduler
        )

        metrics, per_class_acc, _, _ = evaluate(model, val_loader, device)
        val_f1 = metrics['micro']

        current_lr = optimizer.param_groups[1]['lr']
        print(f"Epoch {epoch}: loss={train_loss:.4f}, val_micro_f1={val_f1:.4f}, lr={current_lr:.2e}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            top_checkpoints.append(copy.deepcopy(best_state))
            if len(top_checkpoints) > 5:
                top_checkpoints.pop(0)
            epochs_no_improve = 0
            print(f"  -> New best val Micro-F1: {best_val_f1:.4f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # Average top checkpoints
    if len(top_checkpoints) >= 3:
        avg_state = average_checkpoints(top_checkpoints[-3:])
        model.load_state_dict(avg_state)
        print(f"  Using averaged last 3 best checkpoints")
    elif len(top_checkpoints) >= 1:
        model.load_state_dict(best_state)

    model.to(device)
    print(f"\nBest val Micro-F1: {best_val_f1:.4f}")
    return model, {'best_val_f1': best_val_f1, 'epochs_trained': epoch}


def get_probs(model, loader, device):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Getting probs"):
            batch = to_device(batch, device)
            logits = model(
                report_input_ids=batch['report_input_ids'],
                report_attention_mask=batch['report_attention_mask'],
                clinical=batch['clinical'],
                location=batch['location'],
                location_hierarchy=batch['location_hierarchy'],
                si_per_mod=batch['si_per_mod'],
                si_unknown_mask=batch['si_unknown_mask'],
                cross_mod_ratios=batch['cross_mod_ratios'],
                rad_missing=batch['rad_missing'],
            )
            probs = F.softmax(logits, dim=1)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


print("Training utilities ready (improved for unlimited compute).")
print(f"  LR: main={MAIN_LR:.2e}, BERT={MAIN_LR * BERT_LR_SCALE:.2e}")
print(f"  Label smoothing={LABEL_SMOOTHING}, sqrt weights alpha={NN_WEIGHT_ALPHA}")
print(f"  Augmentation: mask={REPORT_MASK_PROB}, noise={TAB_NOISE_STD}, mixup={MIXUP_ALPHA}")
print(f"  {N_SPLITS}-fold CV, {WARMUP_EPOCHS} warmup epochs")

Training utilities ready (improved for unlimited compute).
  LR: main=3.00e-05, BERT=1.50e-06
  Label smoothing=0.05, sqrt weights alpha=1.0
  Augmentation: mask=0.12, noise=0.1, mixup=0.1
  5-fold CV, 3 warmup epochs


In [ ]:
# ============================================================
# 5-Fold OOF Neural Network Training
#
# Training on train+val combined (2,266 samples)
# 5-fold StratifiedKFold (random_state=42)
# OOF predictions stored for ensemble
# ============================================================
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import ConcatDataset, Subset

print("=" * 70)
print(f"{N_SPLITS}-FOLD OOF NN TRAINING (shift-robust regularization)")
print("=" * 70)

trainval_ds = ConcatDataset([train_ds, val_ds])
trainval_labels = np.concatenate([train_label, val_label])

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_nn_probs = np.zeros((len(trainval_labels), NUM_CLASSES))
oof_nn_preds = np.zeros(len(trainval_labels), dtype=int)
fold_best_f1s = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(range(len(trainval_ds)), trainval_labels)):
    print(f"\nFold {fold + 1}/{N_SPLITS}")

    train_subset = Subset(trainval_ds, tr_idx)
    val_subset = Subset(trainval_ds, va_idx)

    train_fold_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                                   num_workers=0, collate_fn=collate_fn)
    val_fold_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=0, collate_fn=collate_fn)

    fold_model = DualPathBrainTumorClassifier(
        num_classes=NUM_CLASSES,
        si_dim=si_dim,
        num_locations=NUM_LOCATIONS,
        num_regions=NUM_REGIONS,
        num_hemispheres=NUM_HEMISPHERES,
        num_lobes=NUM_LOBES,
        report_dim=REPORT_DIM,
        tabular_dim=TABULAR_DIM,
        freeze_bert_layers=FREEZE_BERT_LAYERS,
        report_dropout=REPORT_DROPOUT,
        tabular_dropout=TABULAR_DROPOUT,
    ).to(device)

    fold_model, fold_info = train_model(
        fold_model,
        train_fold_loader,
        val_fold_loader,
        epochs=300,
        lr=MAIN_LR,
        patience=25,
        weight_alpha=NN_WEIGHT_ALPHA,
        label_smoothing=LABEL_SMOOTHING,
        train_labels_for_weights=trainval_labels[tr_idx],
    )

    fold_probs = get_probs(fold_model, val_fold_loader, device)
    fold_preds = fold_probs.argmax(axis=1)
    fold_f1 = f1_score(trainval_labels[va_idx], fold_preds, average='micro')

    oof_nn_probs[va_idx] = fold_probs
    oof_nn_preds[va_idx] = fold_preds
    fold_best_f1s.append(fold_info['best_val_f1'])
    print(f"  OOF Micro-F1: {fold_f1:.4f} (best epoch: {fold_info['best_val_f1']:.4f})")

    del fold_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

nn_oof_f1 = f1_score(trainval_labels, oof_nn_preds, average='micro')
print(f"\n{'=' * 70}")
print(f"NN {N_SPLITS}-Fold OOF Micro-F1: {nn_oof_f1:.4f}")
print(f"Per-fold best: {[f'{f:.4f}' for f in fold_best_f1s]}")
print(f"{'=' * 70}")

---
## 7. Tree Models (XGBoost, LightGBM, CatBoost)


In [ ]:
!pip install catboost

In [ ]:
# ============================================================
# Tree Models (XGBoost, LightGBM, CatBoost)
#
# 6 models with 5-fold OOF:
#   - NN: Neural network OOF probabilities
#   - NNXGB: XGBoost on NN OOF probs
#   - ImgTab: XGBoost on Image PCA + Tabular features
#   - MLU_XGB: XGBoost on Report SVD + Clinical + Location + Radiomics
#   - MLU_LGB: LightGBM on same features as MLU_XGB
#   - MLU_CB: CatBoost on same features as MLU_XGB
# ============================================================
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler


USE_SQRT_WEIGHTS = False

def get_sqrt_class_weights(y, alpha=0.5):
    """Compute sqrt class weights for soft balancing."""
    classes = np.unique(y)
    cw_balanced = compute_class_weight('balanced', classes=classes, y=y)
    cw_sqrt = np.sqrt(cw_balanced)
    uniform = np.ones_like(cw_sqrt)
    cw_final = (1 - alpha) * uniform + alpha * cw_sqrt
    cw_final = cw_final / cw_final.mean()
    return {int(c): float(w) for c, w in zip(classes, cw_final)}  # No class weights for Micro-F1

# Optimized XGBoost params
MLU_PARAMS = dict(
    n_estimators=3000,
    max_depth=5,
    learning_rate=0.03,
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=5.0,
    verbosity=0,
)

# LightGBM params
MLU_LGB_PARAMS = dict(
    n_estimators=3000,
    max_depth=5,
    learning_rate=0.03,
    objective='multiclass',
    num_class=NUM_CLASSES,
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=5.0,
    verbose=-1,
)

# CatBoost params
MLU_CB_PARAMS = dict(
    iterations=2000,
    depth=5,
    learning_rate=0.05,
    loss_function='MultiClass',
    random_seed=42,
    verbose=0,
)

IMGTAB_PARAMS = dict(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=3.0,
    verbosity=0,
)

NNXGB_PARAMS = dict(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    random_state=42,
    subsample=0.8,
    colsample_bytree=1.0,
    reg_alpha=0.1,
    reg_lambda=1.0,
    verbosity=0,
)

print("="*70)
print("CELL 19: ENHANCED TREE MODELS (5-Fold OOF, NO LEAKAGE)")
print("="*70)

n_train = len(train_label)
n_val = len(val_label)
y_all = np.concatenate([train_label, val_label])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ============================================================
# Report TF-IDF + SVD (improved)
# ============================================================
train_texts = data['train']['report']['report'].fillna('').tolist()
val_texts = data['val']['report']['report'].fillna('').tolist()
test_texts = data['test']['report']['report'].fillna('').tolist()

# Improved TF-IDF: more features, trigrams
tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,3), sublinear_tf=True, min_df=2)
tfidf.fit(train_texts)
svd = TruncatedSVD(n_components=256, random_state=42)
svd.fit(tfidf.transform(train_texts))

# ============================================================
# Feature builders
# ============================================================
def build_loc(split):
    merged = data[split]['merged']
    rows = []
    for _, row in merged.iterrows():
        loc_text = str(row.get('Tumor_Location_raw', ''))
        region, hemi, lobe = parse_location_hierarchy(loc_text)
        rows.append([region, hemi, lobe])
    return np.array(rows, dtype=np.float32)


def build_clin(split):
    merged = data[split]['merged']
    clin_cols = ['Sex_enc', 'Age_clean', 'Age_missing']
    si_cols = [c for c in merged.columns if c.startswith('Signal Intensity')]
    return merged[clin_cols + si_cols].fillna(0).values.astype(np.float32)


def build_rad(split):
    merged = data[split]['merged']
    rad_names = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                 'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
    rows = []
    for _, row in merged.iterrows():
        feats = []
        for mod in MODALITIES:
            for f in rad_names:
                col = f'{mod}_{f}'
                feats.append(float(row.get(col, 0)) if pd.notna(row.get(col)) else 0.0)
            feats.append(float(row.get(f'{mod}_missing', 0)))
        rows.append(feats)
    return np.array(rows, dtype=np.float32)


def build_siunk(split):
    merged = data[split]['merged']
    si_cols_all = [SI_COL_MAPPING[m] for m in MODALITIES]
    rows = []
    for _, row in merged.iterrows():
        cnt = sum(1 for col in si_cols_all if row.get(f'{col}_unknown', 0) == 1)
        rows.append([cnt / 4.0])
    return np.array(rows, dtype=np.float32)


def build_ratios(split):
    merged = data[split]['merged']
    eps = 1e-8
    rad_names = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                 'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
    rows = []
    for _, row in merged.iterrows():
        rad = {}
        for mod in MODALITIES:
            rad[mod] = np.array([row.get(f'{mod}_{f}', 0) for f in rad_names], dtype=np.float32)
        ratios = []
        for i in range(5):
            ratios.extend([
                np.log(np.clip((rad['ax_t1c'][i] + eps) / (rad['ax_t1'][i] + eps), eps, 100)),
                np.log(np.clip((rad['ax_t2f'][i] + eps) / (rad['ax_t2'][i] + eps), eps, 100)),
                np.log(np.clip((rad['ax_t1'][i] + eps) / (rad['ax_t2'][i] + eps), eps, 100)),
                rad['ax_t1c'][i] - rad['ax_t1'][i],
                rad['ax_t2f'][i] - rad['ax_t2'][i],
            ])
        rows.append(ratios)
    return np.array(rows, dtype=np.float32)


def build_imgtab(split):
    """Image PCA (128d x 4 = 512d) + Tabular"""
    merged = data[split]['merged']
    img_dict = data[split]['image']
    eps = 1e-8
    rad_names = ['rad_firstorder_Mean', 'rad_firstorder_Entropy',
                 'rad_firstorder_90Percentile', 'rad_glcm_Contrast', 'rad_glcm_JointEntropy']
    rows = []
    for _, row in merged.iterrows():
        cid = int(row['case_id'])
        img_feats = img_dict.get(cid, {})
        img_vec = np.concatenate([img_feats.get(mod, np.zeros(128, dtype=np.float32)) for mod in MODALITIES])
        clin = row[['Sex_enc', 'Age_clean', 'Age_missing']].values.astype(np.float32)
        rad = {m: np.array([row.get(f'{m}_{f}', 0) for f in rad_names], dtype=np.float32) for m in MODALITIES}
        ratios = []
        for i in range(5):
            ratios.extend([
                np.log(np.clip((rad['ax_t1c'][i] + eps) / (rad['ax_t1'][i] + eps), eps, 100)),
                np.log(np.clip((rad['ax_t2f'][i] + eps) / (rad['ax_t2'][i] + eps), eps, 100)),
                np.log(np.clip((rad['ax_t1'][i] + eps) / (rad['ax_t2'][i] + eps), eps, 100)),
                rad['ax_t1c'][i] - rad['ax_t1'][i],
                rad['ax_t2f'][i] - rad['ax_t2'][i],
            ])
        missing = np.array([1.0 if row.get(f'{m}_missing', 0) else 0.0 for m in MODALITIES], dtype=np.float32)
        loc = np.array([row['Location_enc']], dtype=np.int64)
        si_cols = [c for c in merged.columns if c.startswith('Signal Intensity')]
        si = row[si_cols].values.astype(np.float32)
        rows.append(np.concatenate([img_vec, clin, np.array(ratios, dtype=np.float32), missing, loc, si]))
    return np.array(rows, dtype=np.float32)


def build_mlu(split):
    """Report SVD + Clinical + Location hierarchy + Cross-modal Ratios"""
    merged = data[split]['merged']
    report_lookup = data[split]['report'].set_index('case_id')['report']
    texts = merged['case_id'].map(lambda cid: report_lookup.get(cid, '')).fillna('').tolist()
    x_svd_split = svd.transform(tfidf.transform(texts))
    x_loc = build_loc(split)
    x_clin = build_clin(split)
    x_ratios = build_ratios(split)
    x_siunk = build_siunk(split)
    return np.hstack([x_svd_split, x_clin, x_loc, x_ratios, x_siunk])


print("Building features...")
X_imgtab_all = np.vstack([build_imgtab('train'), build_imgtab('val')])
X_mlu = np.vstack([build_mlu('train'), build_mlu('val')])
print(f"ImgTab: {X_imgtab_all.shape[1]}d | MLU: {X_mlu.shape[1]}d")
print(f"MLU params: {MLU_PARAMS}")

# ============================================================
# Model 1: NN 5-fold OOF probs (from Cell 17)
# ============================================================
print(f"\n[1/6] NN OOF: Micro-F1 = {f1_score(y_all, oof_nn_probs.argmax(axis=1), average='micro'):.4f}")

# ============================================================
# Model 2: NNXGB -- XGB on 5d NN OOF probs
# ============================================================
print("\n[2/6] NNXGB (5-fold OOF):")
oof_nnxgb = np.zeros((len(y_all), NUM_CLASSES))
for fold, (tr_idx, va_idx) in enumerate(skf.split(oof_nn_probs, y_all)):
    clf = xgb.XGBClassifier(**NNXGB_PARAMS)
    if USE_SQRT_WEIGHTS:
        sw = np.array([get_sqrt_class_weights(y_all[tr_idx])[int(y)] for y in y_all[tr_idx]])
        clf.fit(oof_nn_probs[tr_idx], y_all[tr_idx], sample_weight=sw)
    else:
        clf.fit(oof_nn_probs[tr_idx], y_all[tr_idx])
    oof_nnxgb[va_idx] = clf.predict_proba(oof_nn_probs[va_idx])
print(f"  OOF Micro-F1 = {f1_score(y_all, oof_nnxgb.argmax(axis=1), average='micro'):.4f}")

# ============================================================
# Model 3: ImgTab -- Image PCA + Tabular
# ============================================================
print("\n[3/6] ImgTab (5-fold OOF):")
oof_imgtab = np.zeros((len(y_all), NUM_CLASSES))
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_imgtab_all, y_all)):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_imgtab_all[tr_idx])
    X_va = sc.transform(X_imgtab_all[va_idx])
    clf = xgb.XGBClassifier(**IMGTAB_PARAMS)
    if USE_SQRT_WEIGHTS:
        sw = np.array([get_sqrt_class_weights(y_all[tr_idx])[int(y)] for y in y_all[tr_idx]])
        clf.fit(X_tr, y_all[tr_idx], sample_weight=sw)
    else:
        clf.fit(X_tr, y_all[tr_idx])
    oof_imgtab[va_idx] = clf.predict_proba(X_va)
print(f"  OOF Micro-F1 = {f1_score(y_all, oof_imgtab.argmax(axis=1), average='micro'):.4f}")

# ============================================================
# Model 4: MLU -- Report + Clinical + Location + Ratios (XGB)
# ============================================================
print("\n[4/6] MLU-XGB (5-fold OOF):")
oof_mlu_xgb = np.zeros((len(y_all), NUM_CLASSES))
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_mlu, y_all)):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_mlu[tr_idx])
    X_va = sc.transform(X_mlu[va_idx])
    clf = xgb.XGBClassifier(**MLU_PARAMS)
    if USE_SQRT_WEIGHTS:
        sw = np.array([get_sqrt_class_weights(y_all[tr_idx])[int(y)] for y in y_all[tr_idx]])
        clf.fit(X_tr, y_all[tr_idx], sample_weight=sw)
    else:
        clf.fit(X_tr, y_all[tr_idx])
    oof_mlu_xgb[va_idx] = clf.predict_proba(X_va)
print(f"  OOF Micro-F1 = {f1_score(y_all, oof_mlu_xgb.argmax(axis=1), average='micro'):.4f}")

# ============================================================
# Model 5: MLU -- Report + Clinical + Location + Ratios (LGB)
# ============================================================
print("\n[5/6] MLU-LGB (5-fold OOF):")
oof_mlu_lgb = np.zeros((len(y_all), NUM_CLASSES))
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_mlu, y_all)):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_mlu[tr_idx])
    X_va = sc.transform(X_mlu[va_idx])
    clf = lgb.LGBMClassifier(**MLU_LGB_PARAMS)
    if USE_SQRT_WEIGHTS:
        sw = np.array([get_sqrt_class_weights(y_all[tr_idx])[int(y)] for y in y_all[tr_idx]])
        clf.fit(X_tr, y_all[tr_idx], sample_weight=sw)
    else:
        clf.fit(X_tr, y_all[tr_idx])
    oof_mlu_lgb[va_idx] = clf.predict_proba(X_va)
print(f"  OOF Micro-F1 = {f1_score(y_all, oof_mlu_lgb.argmax(axis=1), average='micro'):.4f}")

# ============================================================
# Model 6: MLU -- Report + Clinical + Location + Ratios (CatBoost)
# ============================================================
print("\n[6/6] MLU-CB (5-fold OOF):")
oof_mlu_cb = np.zeros((len(y_all), NUM_CLASSES))
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_mlu, y_all)):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_mlu[tr_idx])
    X_va = sc.transform(X_mlu[va_idx])
    clf = CatBoostClassifier(**MLU_CB_PARAMS)
    if USE_SQRT_WEIGHTS:
        sw = np.array([get_sqrt_class_weights(y_all[tr_idx])[int(y)] for y in y_all[tr_idx]])
        clf.fit(X_tr, y_all[tr_idx], sample_weight=sw)
    else:
        clf.fit(X_tr, y_all[tr_idx])
    oof_mlu_cb[va_idx] = clf.predict_proba(X_va)
print(f"  OOF Micro-F1 = {f1_score(y_all, oof_mlu_cb.argmax(axis=1), average='micro'):.4f}")

# ============================================================
# Summary
# ============================================================
print("\n" + "="*70)
print("CLEAN OOF SUMMARY")
print("="*70)
nn_f1 = f1_score(y_all, oof_nn_probs.argmax(axis=1), average='micro')
nn_macro = f1_score(y_all, oof_nn_probs.argmax(axis=1), average='macro')
nnxgb_f1 = f1_score(y_all, oof_nnxgb.argmax(axis=1), average='micro')
nnxgb_macro = f1_score(y_all, oof_nnxgb.argmax(axis=1), average='macro')
imgtab_f1 = f1_score(y_all, oof_imgtab.argmax(axis=1), average='micro')
imgtab_macro = f1_score(y_all, oof_imgtab.argmax(axis=1), average='macro')
mlu_xgb_f1 = f1_score(y_all, oof_mlu_xgb.argmax(axis=1), average='micro')
mlu_xgb_macro = f1_score(y_all, oof_mlu_xgb.argmax(axis=1), average='macro')
mlu_lgb_f1 = f1_score(y_all, oof_mlu_lgb.argmax(axis=1), average='micro')
mlu_lgb_macro = f1_score(y_all, oof_mlu_lgb.argmax(axis=1), average='macro')
mlu_cb_f1 = f1_score(y_all, oof_mlu_cb.argmax(axis=1), average='micro')
mlu_cb_macro = f1_score(y_all, oof_mlu_cb.argmax(axis=1), average='macro')

print(f"  {'Model':<12} {'Micro-F1':>10} {'Macro-F1':>10}")
print(f"  {'-'*35}")
print(f"  {'NN':<12} {nn_f1:>10.4f} {nn_macro:>10.4f}")
print(f"  {'NNXGB':<12} {nnxgb_f1:>10.4f} {nnxgb_macro:>10.4f}")
print(f"  {'ImgTab':<12} {imgtab_f1:>10.4f} {imgtab_macro:>10.4f}")
print(f"  {'MLU-XGB':<12} {mlu_xgb_f1:>10.4f} {mlu_xgb_macro:>10.4f}")
print(f"  {'MLU-LGB':<12} {mlu_lgb_f1:>10.4f} {mlu_lgb_macro:>10.4f}")
print(f"  {'MLU-CB':<12} {mlu_cb_f1:>10.4f} {mlu_cb_macro:>10.4f}")

In [ ]:
# ============================================================
# Train Final Models for Test Prediction
#
# Trains:
#   - Global NN model on all data (10-fold split for val)
#   - XGBoost on NN OOF probabilities
#   - ImgTab: XGBoost on Image + Tabular
#   - MLU: XGBoost, LightGBM, CatBoost on Report + Clinical + Location
# ============================================================
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

def get_sqrt_class_weights(y, alpha=0.5):
    """Compute sqrt class weights for soft balancing."""
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(y)
    cw_balanced = compute_class_weight('balanced', classes=classes, y=y)
    cw_sqrt = np.sqrt(cw_balanced)
    uniform = np.ones_like(cw_sqrt)
    cw_final = (1 - alpha) * uniform + alpha * cw_sqrt
    cw_final = cw_final / cw_final.mean()
    return {int(c): float(w) for c, w in zip(classes, cw_final)}



print("=" * 70)
print("CELL 22: Final Models for Test Prediction")
print("=" * 70)

# NN: train global model for test probs
global_ds = ConcatDataset([train_ds, val_ds])
n_train = len(train_label)
all_labels = np.concatenate([train_label, val_label])
gskf = StratifiedKFold(n_splits=10, shuffle=True, random_state=999)
all_idx = np.arange(len(all_labels))
tr_idx, va_idx = next(gskf.split(all_idx, all_labels))

global_model = DualPathBrainTumorClassifier(
    num_classes=NUM_CLASSES,
    si_dim=si_dim,
    num_locations=NUM_LOCATIONS,
    num_regions=NUM_REGIONS,
    num_hemispheres=NUM_HEMISPHERES,
    num_lobes=NUM_LOBES,
    report_dim=REPORT_DIM,
    tabular_dim=TABULAR_DIM,
    freeze_bert_layers=FREEZE_BERT_LAYERS,
    report_dropout=REPORT_DROPOUT,
    tabular_dropout=TABULAR_DROPOUT,
).to(device)

global_model, global_info = train_model(
    global_model,
    DataLoader(Subset(global_ds, tr_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn),
    DataLoader(Subset(global_ds, va_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn),
    epochs=300,
    lr=MAIN_LR,
    patience=30,
    weight_alpha=NN_WEIGHT_ALPHA,
    label_smoothing=LABEL_SMOOTHING,
)
print(f"Global model best val F1: {global_info['best_val_f1']:.4f}")
nn_test_probs = get_probs(global_model, test_loader, device)
print(f"NN test probs: {nn_test_probs.shape}")
del global_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# NNXGB: retrain on full train OOF probs -> predict on nn_test_probs
nnxgb_test = xgb.XGBClassifier(**NNXGB_PARAMS)
nnxgb_test.fit(oof_nn_probs, np.concatenate([train_label, val_label]))
nnxgb_test_probs = nnxgb_test.predict_proba(nn_test_probs)
print(f"NNXGB test probs: {nnxgb_test_probs.shape}")

# ImgTab: retrain on full train+val, predict on test
imgtab_test = xgb.XGBClassifier(**IMGTAB_PARAMS)
imgtab_test.fit(X_imgtab_all, np.concatenate([train_label, val_label]))
imgtab_test_probs = imgtab_test.predict_proba(build_imgtab('test'))
print(f"ImgTab test probs: {imgtab_test_probs.shape}")

# MLU-XGB: train on full train+val, predict on test
mlu_xgb_test = xgb.XGBClassifier(**MLU_PARAMS)
mlu_xgb_test.fit(X_mlu, np.concatenate([train_label, val_label]))
mlu_xgb_test_probs = mlu_xgb_test.predict_proba(build_mlu('test'))
print(f"MLU-XGB test probs: {mlu_xgb_test_probs.shape}")

# MLU-LGB: train on full train+val, predict on test
mlu_lgb_test = lgb.LGBMClassifier(**MLU_LGB_PARAMS)
mlu_lgb_test.fit(X_mlu, np.concatenate([train_label, val_label]))
mlu_lgb_test_probs = mlu_lgb_test.predict_proba(build_mlu('test'))
print(f"MLU-LGB test probs: {mlu_lgb_test_probs.shape}")

# MLU-CB: train on full train+val, predict on test
mlu_cb_test = CatBoostClassifier(**MLU_CB_PARAMS)
mlu_cb_test.fit(X_mlu, np.concatenate([train_label, val_label]))
mlu_cb_test_probs = mlu_cb_test.predict_proba(build_mlu('test'))
print(f"MLU-CB test probs: {mlu_cb_test_probs.shape}")

print("\nAll test probs ready for Cell 27:")
print(f"  nn_test_probs:      {nn_test_probs.shape}")
print(f"  nnxgb_test_probs:  {nnxgb_test_probs.shape}")
print(f"  imgtab_test_probs: {imgtab_test_probs.shape}")
print(f"  mlu_xgb_test:     {mlu_xgb_test_probs.shape}")
print(f"  mlu_lgb_test:     {mlu_lgb_test_probs.shape}")
print(f"  mlu_cb_test:      {mlu_cb_test_probs.shape}")

In [ ]:
# ============================================================
# Ensemble & Submission
#
# Combines 6 models via weighted average:
#   - Grid search for optimal weights on OOF predictions
#   - Coarse search (step=0.1) + Fine search (step=0.05)
#
# Best model: nn_040_mluxgb_030_mlulgb_020_nnxgb_010
# Weights: NN=0.4, MLU_XGB=0.3, MLU_LGB=0.2, NNXGB=0.1
# ============================================================
import itertools
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize

print("="*70)
print("CELL 27: Advanced Ensemble & Submission")
print("="*70)

y_all = np.concatenate([train_label, val_label])

# 6 models from Cell 19
models_oof = {
    'NN':      oof_nn_probs,
    'NNXGB':   oof_nnxgb,
    'ImgTab':  oof_imgtab,
    'MLU_XGB': oof_mlu_xgb,
    'MLU_LGB': oof_mlu_lgb,
    'MLU_CB':  oof_mlu_cb,
}

models_test = {
    'NN':      nn_test_probs,
    'NNXGB':   nnxgb_test_probs,
    'ImgTab':  imgtab_test_probs,
    'MLU_XGB': mlu_xgb_test_probs,
    'MLU_LGB': mlu_lgb_test_probs,
    'MLU_CB':  mlu_cb_test_probs,
}

print("\nIndividual OOF Scores:")
for name, probs in models_oof.items():
    f1 = f1_score(y_all, probs.argmax(axis=1), average='micro')
    print(f"  {name:<10}: Micro-F1 = {f1:.4f}")

# ============================================================
# Grid Search Blend (coarse + fine)
# ============================================================
print("\n" + "-"*50)
print("Grid Search (step=0.1)")

best_f1, best_weights = 0, None
model_names = list(models_oof.keys())

# Simple grid for 6 models (reduced search space)
for w_nn in [0.2, 0.3, 0.4]:
    for w_mlu_xgb in [0.2, 0.3, 0.4]:
        for w_mlu_lgb in [0.1, 0.2, 0.3]:
            w_rest = 1.0 - w_nn - w_mlu_xgb - w_mlu_lgb
            if w_rest < 0 or w_rest > 0.3:
                continue
            weights = {'NN': w_nn, 'NNXGB': 0.05, 'ImgTab': 0.05,
                      'MLU_XGB': w_mlu_xgb, 'MLU_LGB': w_mlu_lgb, 'MLU_CB': w_rest}
            blend = sum(w * models_oof[k] for k, w in weights.items())
            f1 = f1_score(y_all, blend.argmax(axis=1), average='micro')
            if f1 > best_f1:
                best_f1, best_weights = f1, weights.copy()

print(f"Best coarse: {best_f1:.4f}")
print(f"Best weights: {best_weights}")

# Fine-tune around best
print("\nFine-tuning (step=0.05)...")
for delta in [-0.05, 0, 0.05]:
    for k in ['NN', 'MLU_XGB', 'MLU_LGB']:
        new_weights = best_weights.copy()
        new_weights[k] = max(0, best_weights[k] + delta)
        total = sum(new_weights.values())
        for key in new_weights:
            new_weights[key] /= total
        blend = sum(w * models_oof[k] for k, w in new_weights.items())
        f1 = f1_score(y_all, blend.argmax(axis=1), average='micro')
        if f1 > best_f1:
            best_f1, best_weights = f1, new_weights.copy()

print(f"Best fine: {best_f1:.4f}")
print(f"Final weights: {best_weights}")

# ============================================================
# Probability Blend (use best weights from grid search)
# ============================================================
print("\n" + "="*70)
print("GENERATING SUBMISSION")
print("="*70)

print(f"Using probability blend: OOF Micro-F1 = {best_f1:.4f}")
print(f"Weights: {best_weights}")

# Apply weights to test probabilities
final_test_probs = sum(best_weights[k] * models_test[k] for k in model_names)

final_preds = final_test_probs.argmax(axis=1)
final_labels = label_encoder.inverse_transform(final_preds)

submission = pd.DataFrame({
    'case_id': data['test']['merged']['case_id'].values,
    'Overall_class': final_labels
})
submission.to_csv('submission_final.csv', index=False)
print(f"\nSubmission saved: {len(submission)} predictions")
print(submission['Overall_class'].value_counts())

In [ ]:
# ============================================================
# Robust submission sweep
# Generates multiple submission files for Kaggle probing
# ============================================================

# Final submission configurations
# Best model (OOF=87.33%): nn_040_mluxgb_030_mlulgb_020_nnxgb_010
SUBMISSION_CONFIGS = {
    # Best model: NN-dominant blend
    'final': {
        'NN': 0.40,
        'NNXGB': 0.05,
        'ImgTab': 0.05,
        'MLU_XGB': 0.30,
        'MLU_LGB': 0.20,
        'MLU_CB': 0.00,
    },
    # Alternative: More MLU weight
    'mlu_heavy': {
        'NN': 0.40,
        'NNXGB': 0.05,
        'ImgTab': 0.05,
        'MLU_XGB': 0.30,
        'MLU_LGB': 0.10,
        'MLU_CB': 0.10,
    },
    # Pure NN (max robustness to distribution shift)
    'nn_only': {
        'NN': 1.00,
        'NNXGB': 0.00,
        'ImgTab': 0.00,
        'MLU_XGB': 0.00,
        'MLU_LGB': 0.00,
        'MLU_CB': 0.00,
    },
    # Pure MLU (max in-distribution accuracy)
    'mlu_only': {
        'NN': 0.00,
        'NNXGB': 0.00,
        'ImgTab': 0.00,
        'MLU_XGB': 0.50,
        'MLU_LGB': 0.30,
        'MLU_CB': 0.20,
    },
}

# Normalize any weights that don't sum to 1.0
for name, weights in SUBMISSION_CONFIGS.items():
    total = sum(weights.values())
    if abs(total - 1.0) > 1e-6:
        SUBMISSION_CONFIGS[name] = {k: v/total for k, v in weights.items()}
        print(f"Normalized {name}: {total:.4f} -> 1.0")

print("\nRobust submission sweep:")
for name, weights in SUBMISSION_CONFIGS.items():
    total = sum(weights.values())
    assert abs(total - 1.0) < 1e-6, f"{name} weights sum to {total}"

    blend_test = sum(weights[k] * models_test[k] for k in weights.keys())
    preds = blend_test.argmax(axis=1)
    labels = label_encoder.inverse_transform(preds)

    blend_oof = sum(weights[k] * models_oof[k] for k in weights.keys())
    f1 = f1_score(y_all, blend_oof.argmax(axis=1), average='micro')

    sub = pd.DataFrame({'case_id': test_case_ids, 'Overall_class': labels})
    fname = f'submission_{name}.csv'
    sub.to_csv(fname, index=False)
    print(f"  {name}: OOF={f1:.4f} | dist: {dict(sub['Overall_class'].value_counts())}")


#We chose submission_mlu_heavy.csv and submission_final.csv in Kaggle